In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

RNG = np.random.default_rng(42)
plt.rcParams["figure.dpi"] = 110
sns.set_theme(style="whitegrid", palette="deep")

D = np.load("bsds_pixels.npz", allow_pickle=True)
X_tr, y_tr = D["X_tr"].astype(float), D["y_tr"].astype(int)
X_te, y_te = D["X_te"].astype(float), D["y_te"].astype(int)
features = list(D["features"])

print("train:", X_tr.shape, " test:", X_te.shape)
print("boundary rate  train:", round(float(y_tr.mean()),4), " test:", round(float(y_te.mean()),4))
pd.DataFrame(X_tr, columns=features).describe().T.round(2)
from sklearn.metrics import f1_score, accuracy_score, roc_auc_score

train: (48000, 17)  test: (18000, 17)
boundary rate  train: 0.2667  test: 0.2667


In [2]:
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
from sklearn.metrics import classification_report, roc_auc_score

def slog(x):  # signed log1p — a/b channels can be negative
    return np.sign(x) * np.log1p(np.abs(x))

gnb_raw = GaussianNB().fit(X_tr, y_tr)
gnb_log = GaussianNB().fit(slog(X_tr), y_tr)
qda_log = QuadraticDiscriminantAnalysis(reg_param=1e-3).fit(slog(X_tr), y_tr)

print("GaussianNB raw : AUC =", round(roc_auc_score(y_te, gnb_raw.predict_proba(X_te)[:,1]), 4))
print("GaussianNB slog: AUC =", round(roc_auc_score(y_te, gnb_log.predict_proba(slog(X_te))[:,1]), 4))
print("QDA slog (independence assumption removed): AUC =",
      round(roc_auc_score(y_te, qda_log.predict_proba(slog(X_te))[:,1]), 4))
print()
print(classification_report(y_te, gnb_log.predict(slog(X_te)), target_names=["non-boundary","boundary"]))

GaussianNB raw : AUC = 0.8191
GaussianNB slog: AUC = 0.8246
QDA slog (independence assumption removed): AUC = 0.832

              precision    recall  f1-score   support

non-boundary       0.87      0.81      0.84     13200
    boundary       0.56      0.67      0.61      4800

    accuracy                           0.77     18000
   macro avg       0.71      0.74      0.72     18000
weighted avg       0.79      0.77      0.78     18000

